## Laboratorio 3: Detección de malware

Primero vamos a determinar si los archivos están empaquetados

In [17]:
import pefile
import os
import subprocess
import pandas as pd
import datetime
def show_sections(pefile_object):
    for section in pefile_object.sections:
        print("Nombre:", section.Name)
        print("VirtualAddress:", hex(section.VirtualAddress))
        print("Size:", section.SizeOfRawData)
        print("-" * 20)


carpeta = "MALWR"
archivos_procesados = 0

for archivo in os.listdir(carpeta):
    ruta = os.path.join(carpeta, archivo)

    # Solo archivos normales
    if os.path.isfile(ruta):
        try:
            print("\n" + "="*60)
            print(f"Analizando: {archivo}")
            print("="*60)

            pe = pefile.PE(ruta)
            show_sections(pe)

            archivos_procesados += 1

            if archivos_procesados >= 5:
                break

        except Exception as e:
            print(f"No es un PE válido: {archivo}")

if archivos_procesados == 0:
    print("No se encontraron ejecutables PE válidos.")



Analizando: F6655E39465C2FF5B016980D918EA028
Nombre: b'.text\x00\x00\x00'
VirtualAddress: 0x1000
Size: 3584
--------------------
Nombre: b'.rdata\x00\x00'
VirtualAddress: 0x2000
Size: 1536
--------------------
Nombre: b'.data\x00\x00\x00'
VirtualAddress: 0x3000
Size: 512
--------------------
Nombre: b'.rsrc\x00\x00\x00'
VirtualAddress: 0x4000
Size: 512
--------------------

Analizando: FGJKJJ1_2BA0D0083976A5C1E3315413CDCFFCD2
Nombre: b'.text\x00\x00\x00'
VirtualAddress: 0x1000
Size: 4096
--------------------
Nombre: b'.rdata\x00\x00'
VirtualAddress: 0x2000
Size: 2048
--------------------
Nombre: b'.data\x00\x00\x00'
VirtualAddress: 0x3000
Size: 512
--------------------
Nombre: b'.rsrc\x00\x00\x00'
VirtualAddress: 0x4000
Size: 512
--------------------

Analizando: 65018CD542145A3792BA09985734C12A
Nombre: b'.text\x00\x00\x00'
VirtualAddress: 0x1000
Size: 4096
--------------------
Nombre: b'.rdata\x00\x00'
VirtualAddress: 0x2000
Size: 2048
--------------------
Nombre: b'.data\x00\x00\x00

Vemos que si están empaquetados, vamos a desempaquetarlos

In [5]:
for file in os.listdir('MALWR'):
    route = os.path.join('MALWR', file)
    result = subprocess.run(['upx', '-d', route], capture_output=True, text=True)
    
    if result.returncode == 0:
        print(f' {file} comprimido')
    else:
        print(f'{file} falló: {result.stderr}')

 F6655E39465C2FF5B016980D918EA028 comprimido
 FGJKJJ1_2BA0D0083976A5C1E3315413CDCFFCD2 comprimido
 65018CD542145A3792BA09985734C12A comprimido
 6FAA4740F99408D4D2DDDD0B09BBDEFD comprimido
 F8437E44748D2C3FCF84019766F4E6DC comprimido
 B98hX8E8622C393D7E832D39E620EAD5D3B49 comprimido
SAM_B659D71AE168E774FAAF38DB30F4A84 falló: upx: MALWR/SAM_B659D71AE168E774FAAF38DB30F4A84: NotPackedException: not packed by UPX

POL55_A4F1ECC4D25B33395196B5D51A06790 falló: upx: MALWR/POL55_A4F1ECC4D25B33395196B5D51A06790: NotPackedException: not packed by UPX

 BVJ2D9FBF759F527AF373E34673DC3ACA462 comprimido
 AAAz2E1B6940985A23E5639450F8391820655 comprimido
 VBMM9_149B7BD7218AAB4E257D28469FDDB0D comprimido
 NBV_8B75BCBFF174C25A0161F30758509A44 comprimido
 TG78Z__727A6800991EEAD454E53E8AF164A99C comprimido
 RTC_7F85D7F628CE62D1D8F7B39D8940472 comprimido
 B07322743778B5868475DBE66EEDAC4F comprimido
 AL65_DB05DF0498B59B42A8E493CF3C10C578 comprimido
 JKK8CA6FE7A1315AF5AFEAC2961460A80569 comprimido
 FHHH6576C1

Ya desempaquetados, vamos a crear el dataset, este tendrá estas características:

- Hash del archivop
- Imports
- Timestamp de compliación


In [27]:
def get_imports(path):
    """
    Extract function calls from header
    """
    imports = set()
    timestamp = None
    try:
        pe = pefile.PE(path)
        timestamp = pe.FILE_HEADER.TimeDateStamp

        if hasattr(pe, "DIRECTORY_ENTRY_IMPORT"):
            for entry in pe.DIRECTORY_ENTRY_IMPORT:
                dll = entry.dll.decode(errors="ignore")

                for imp in entry.imports:
                    if imp.name:
                        func = imp.name.decode(errors="ignore")
                        imports.add(f"{dll}:{func}")

    except pefile.PEFormatError:
        return set(), None

    except Exception:
        return set(), None
    
    return imports, timestamp


df =  pd.DataFrame(columns=['hash', 'imports', 'timestamp'])

for file in os.listdir('MALWR'):
    route = os.path.join('MALWR', file)
    imports, timestamp = get_imports(route)

    imports = ' '.join(list(imports))

    new_row = pd.DataFrame([{'hash': file, 'imports': imports, 'timestamp': timestamp }])

    df = pd.concat([df, new_row], ignore_index=True)
    
    if result.returncode == 0:
        print(f' {file} comprimido')
    else:
        print(f'{file} falló: {result.stderr}')


df.to_csv('raw_data.csv')

 F6655E39465C2FF5B016980D918EA028 comprimido
 FGJKJJ1_2BA0D0083976A5C1E3315413CDCFFCD2 comprimido
 65018CD542145A3792BA09985734C12A comprimido
 6FAA4740F99408D4D2DDDD0B09BBDEFD comprimido
 F8437E44748D2C3FCF84019766F4E6DC comprimido
 B98hX8E8622C393D7E832D39E620EAD5D3B49 comprimido
 SAM_B659D71AE168E774FAAF38DB30F4A84 comprimido
 POL55_A4F1ECC4D25B33395196B5D51A06790 comprimido
 BVJ2D9FBF759F527AF373E34673DC3ACA462 comprimido
 AAAz2E1B6940985A23E5639450F8391820655 comprimido
 VBMM9_149B7BD7218AAB4E257D28469FDDB0D comprimido
 NBV_8B75BCBFF174C25A0161F30758509A44 comprimido
 TG78Z__727A6800991EEAD454E53E8AF164A99C comprimido
 RTC_7F85D7F628CE62D1D8F7B39D8940472 comprimido
 B07322743778B5868475DBE66EEDAC4F comprimido
 AL65_DB05DF0498B59B42A8E493CF3C10C578 comprimido
 JKK8CA6FE7A1315AF5AFEAC2961460A80569 comprimido
 FHHH6576C196385407B0F7F4B1B537D88983 comprimido
 785003A405BC7A4EBCBB21DDB757BF3F comprimido
 L11_1415EB8519D13328091CC5C76A624E3D comprimido
 FTTR9EA3C16194CE354C244C1B74C46CD